## Extract and process Germline data from ClinVar

Search for the term 'USER' to locate codes lines where user must provide input

In [1]:
# 1) import modules
import os
import re
import json
import pandas as pd
import scipy
import time
import requests
import hashlib
import csv
import random
from collections import defaultdict
import numpy as np    
import statsmodels.api as sm   
import seaborn as sns
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

In [ ]:
# 2) extract germline data from ClinVar using e-utilities

##USER INPUT NEEDED: 
# i) Path to folder where extracted files will be stored (should contain folders for each gene of interest and file with gene information needed in ii)
# ii)compiled list of gene symbols + gene ID (NCBI) of interest in a .txt file to read into 'TranscriptID' (Supplemental Table)
## Manually compiled this file in the parent directory: file used for this analysis: 'TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt' \
# that contains Gene symbols + Gene ID to load into the GID variable when using commandline command to extract data\
# using e-utilities via EDirect

start_time_block2=time.asctime(time.localtime())
print(start_time_block2)

#i) 
parentpath="USER INSERT PARENT PATH TO FOLDER WHERE YOU WILL CREATE SUB-FOLDERS PER GENE OF INTEREST AND OUTPUT EXTRACTED DATA"

os.chdir(parentpath)

#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4 for this study) and Gene ID
#ii) USER INPUT FILE NAME 
TranscriptID= pd.read_csv("USER INPUT FILE NAME CONTAINING GENE SYMBOL+NCBI GENE ID+ MANE TRANSCRIPT ID (Supplemental Table S4)", sep="\t")

# Loop through each folder name aka each gene in the list to read the respective extracted variant file 
for index, row in TranscriptID.iterrows():
    #initialize variable with gene symbol
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    
    #if separate folder for each gene not already created in parent path, create it now:
    if os.path.exists(folder_name)==False:
        os.mkdir(folder_name)
    
    #if 'folder_name' folder exists, list all files in that folder then change directory to this folder (to go to each gene's folder and read+write files for each gene individually)
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        os.chdir(folder_name)
        print(folder_name)
        #print code progress as the gene symbol or folder name currently being worked on:
        print("Current directory:", os.getcwd())
        #load Gene ID to the variable that will be used in the e-utilities command via Entrez Direct beginning with "!" below
        GIDvar=TranscriptID.loc[index]["GeneID"]
        print(GIDvar)
        
        #e-utilities command: 
        # esearch will search through the database (clinvar in this case), querying for any entry with the Gene ID\
        # loaded to GID variable
        
        # efetch will then get all information by variationid for variants in genes with gene ID queried
        
        # efetch gets records in xml format, xtract will take pieces from the xml format and create a \
        # tab-delimited (default) file with the elements defined 
        
        # for example, here the parent of the xml tree is VariationArchive- then I called for the entry of Gene@Symbol \
        # that is a child element in this tree -so the output first column will be Gene@Symbol - this should be the \
        # same for all rows in the file (every row represents a variant)
        
        # when the GeneID is equal to GeneID entered in the variable- this if statement was added to include only \
        # those variants that affect the gene of interest and remove info on other genes if they overlap at the locus of the variant
        
        # tab-delimited columns thus extracted are:
        ## 1. Gene@Symbol (QC to make sure all vars are in the gene queried)
        ## 2. Gene@GeneID
        ## 3. VariationID (clinvar id)
        ## 4. VariationName 
        ## 5. VariationType 
        ## 6. NumberOfSubmissions 
        ## 7. NumberOfSubmitters
        ## 8. GermlineClassification@NumberOfSubmissions (new added when they released the somatic classification update this year) _ \
        ## 9. GermlineClassification@NumberOfSubmitters
        ## 10. ReviewStatus (1 star, etc)
        ## 11. Description (first in the xml file)- (I have a question here- Description contains P/LP/VUS/LB/B info: the oevrall classification.\
        ## 12. Explanation (for conflicting variants- this element has the break up of # submissions and their respective classifications, eg: Pathogenic(10);Benign(1) - 10 P submissions and 1 B submission)
        ## 13. ClinVarAccession@SubmitterName,ClinVarAccession@OrgID,Classification@DateLastEvaluated,ReviewStatus,GermlineClassification,Origin   \
        # (This element is "|" seperated for all the ',' separated terms here such that all submission wise information is displayed back to back in 1 column - easy to put into a python dictionary \
        ## 14. SimpleAllele@AlleleID (clinvar allele ID - has no use except to be a filler in the column to have the next element tab separated)
        ## 15. Expression (HGVSc on MANE Select if exists; or else HGVSg on GRCh38)
        ## 16. Assembly (On which the HGVS above is called- QC to make sure no GRCh37 variants in the dataset)
        
        !esearch -db clinvar -query "$GIDvar[GID]"|efetch -format variationid| xtract -pattern VariationArchive -group Gene -if Gene@GeneID -equals "$GIDvar" -element Gene@Symbol Gene@GeneID -group VariationArchive\
        -element VariationArchive@VariationID VariationArchive@VariationName VariationArchive@VariationType VariationArchive@NumberOfSubmissions VariationArchive@NumberOfSubmitters -group Classifications -subset GermlineClassification \
        -element GermlineClassification@NumberOfSubmissions GermlineClassification@NumberOfSubmitters ReviewStatus -first Description -element Explanation -group ClinicalAssertionList -subset ClinicalAssertion -tab "/" -sep "|" -element \
        ClinVarAccession@SubmitterName,ClinVarAccession@OrgID,Classification@DateLastEvaluated,ReviewStatus,GermlineClassification,Origin -group SimpleAllele -tab "\t" -sep "|" -element SimpleAllele@AlleleID -group HGVSlist -block \
        NucleotideExpression -if NucleotideExpression@MANESelect -element Expression -block HGVS -if HGVS@Assembly -equals "GRCh38" -element Expression HGVS@Assembly > ClinVarAPI_xtract_data.txt
        
        time_now=time.asctime(time.localtime())
        print(time_now)
        
        #checkpoint for where the code is while running- print gene name (directory)
        print("Current directory:", os.getcwd())
        #go back to parent directory after completing gene and start over with next gene
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)



In [2]:
# 3) filter and process germline data 

##USER INPUT NEEDED: 
# i) Path to folder where extracted files will be stored (should contain folders for each gene of interest and file with gene information needed in ii)
# ii)compiled list of gene symbols + gene ID (NCBI) of interest in a .txt file to read into 'TranscriptID'

#time at start and end to record date and time code was run + measure how long the code takes to run
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)

#initialize dataframe to compile variant counts at each filter/processing step:
allTSGdfcounts=pd.DataFrame()

#i) 
parentpath="USER INSERT PARENT PATH TO FOLDER WHERE YOU WILL CREATE SUB-FOLDERS PER GENE OF INTEREST AND OUTPUT EXTRACTED DATA"

os.chdir(parentpath)

#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4 for this study) and Gene ID
#ii) USER INPUT FILE NAME 
TranscriptID= pd.read_csv("USER INPUT FILE NAME CONTAINING GENE SYMBOL+NCBI GENE ID+ MANE SELECT TRANSCRIPT ID (Supplemental Table)", sep="\t")

# Loop through each folder name aka each gene in the list to read the respective extracted variant file 
for index, row in TranscriptID.iterrows():
    
    #initialize variable with gene symbol
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    
    #if folder name exists, list all files in that folder then change directory to that folder (to go to each gene's folder and read+write files for each gene individually)
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        print(folder_name)
        #print code progress as the gene symbol or folder name currently being worked on:
        print("Current directory:", os.getcwd())
        os.chdir("USER INSERT PATH")
        
        # read extracted file from ClinVar with the column names as defined here- xtract will not automatically insert colnames so we are defining them here:
        inputfile_clinvar_colnames = ["GeneSymbol", "GeneID", "VariationID", "VariationName", "VariationType","NumberOfSubmissions", "NumberOfSubmitters","GermlineClassificationNumberOfSubmissions","GermlineClassificationNumberOfSubmitters", "ReviewStatus", "GL_first_Description", "Explanation", "SubmissionList", "HGVS", "Assembly"]
        #BRCA1 and BRCA2 run on 6-17:
        #if (folder_name=="BRCA1") or (folder_name=="BRCA2"):
        #    clinvar_rawdata = pd.read_csv("ClinVarAPI_xtract_update_6-17-24.txt", sep="\t", names=inputfile_clinvar_colnames, header=None, usecols=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14])
        #else:
        clinvar_rawdata = pd.read_csv("ClinVarAPI_xtract_data.txt", sep="\t", names=inputfile_clinvar_colnames, header=None, usecols=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14])
        #clinvar_rawdata = pd.read_csv("ClinVarAPI_xtract_data.txt", sep="\t", names=inputfile_clinvar_colnames, header=None, usecols=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14])
        #counting and printing variants at every filter/process step for quality control:

        One_allclinvar=len(clinvar_rawdata)
        
        #filter variation type
        Variation_type_toexclude = ["Complex", "Translocation", "Variation", "fusion", "protein only", "Haplotype", "copy number gain", "copy number loss"]
        clinvar_rawdata_vartypefilter = clinvar_rawdata[~clinvar_rawdata['VariationType'].isin(Variation_type_toexclude)]
        Two_clinvarvartype=len(clinvar_rawdata_vartypefilter)
        
        
        #filter conflicting variants:
        clinvar_rawdata_vartype_reviewstatus_conflict= clinvar_rawdata_vartypefilter[clinvar_rawdata_vartypefilter['GL_first_Description'].str.contains("Conflicting classifications of pathogenicity")]
        #rename colnames for conflicting df to match with p/lp so that can concate conflicting variants scoring>75 with p/lp df
        clinvar_rawdata_vartype_reviewstatus_conflict_rename=clinvar_rawdata_vartype_reviewstatus_conflict.rename(columns={"HGVS":"HGVSc", "Assembly":"HGVSg"})
        
        clinvar_rawdata_vartype_reviewstatus_PLP=clinvar_rawdata_vartypefilter.drop(clinvar_rawdata_vartype_reviewstatus_conflict.index)
        #rename colnames for PLP df (since these variants dont have an 'Explanation' column- that col is present only for conflicting variants, but eutils xtract mushes all info together and doesnt demarcate sep cols)
        clinvar_rawdata_vartype_reviewstatus_PLP_rename=clinvar_rawdata_vartype_reviewstatus_PLP.rename(columns={"Explanation":"PLP_SubmissionList", "SubmissionList":"PLP_HGVSc", "HGVS":"PLP_HGVSg"})
        #have to do it in 2 steps since cant have 2 cols with same name
        clinvar_rawdata_vartype_reviewstatus_PLP_rename=clinvar_rawdata_vartype_reviewstatus_PLP_rename.rename(columns={"PLP_SubmissionList":"SubmissionList", "PLP_HGVSc":"HGVSc", "PLP_HGVSg":"HGVSg"})

        ### CONFLICT SCORE ###
        #calculate score for conflicting variants  - higher scoring variants have a greater proportion of PLP submissions and added back in with PLP variants if they scored greater than or equal to 75
        #adding col for score
        clinvar_rawdata_vartype_reviewstatus_conflict_rename.insert(1,"Score","NaN")
        for index, row in clinvar_rawdata_vartype_reviewstatus_conflict_rename.iterrows():
            rowindex=clinvar_rawdata_vartype_reviewstatus_conflict_rename.index.get_loc(index)
            colindex=clinvar_rawdata_vartype_reviewstatus_conflict_rename.columns.get_loc('Score')
            
            colindex_interpretation=clinvar_rawdata_vartype_reviewstatus_conflict_rename.columns.get_loc('GL_first_Description')
                
            #initialize counts to 0 for every row (aka variant)
            count_vus=count_p=count_lp=count_b=count_lb=score_vus=score_p=score_lp=score_b=score_lb=int(0)
            #print(clinvar_rawdata_vartype_reviewstatus_conflict_rename.loc[index]["Explanation"])
            explanation=str(clinvar_rawdata_vartype_reviewstatus_conflict_rename.loc[index]["Explanation"]).split("; ")
            for item in explanation:
                if item.startswith("Uncertain significance"):
                    valueip=item.split("(")
                    valueip2=valueip[1].split(")")
                    count_vus=count_vus+int(valueip2[0])
                    
                else:
                    if item.startswith("Pathogenic"):
                        valueip=item.split("(")
                        valueip2=valueip[1].split(")")
                        count_p=count_p+int(valueip2[0])
                        
                    else:
                        if item.startswith("Likely pathogenic"):
                            valueip=item.split("(")
                            valueip2=valueip[1].split(")")
                            count_lp=valueip2[0]
                            
                        else:
                            if item.startswith("Likely benign"):
                                valueip=item.split("(")
                                valueip2=valueip[1].split(")")
                                count_lb=valueip2[0]
                                
                            else:
                                if item.startswith("Benign"):
                                    valueip=item.split("(")
                                    valueip2=valueip[1].split(")")
                                    count_b=valueip2[0]
                                    
            #multiply number entries for each classification with median posterior probability of that classification
            score_vus=(50*int(count_vus))
            score_p=(99.5*int(count_p))
            score_lp=(95*int(count_lp))
            score_lb=(5*int(count_lb))
            score_b=(0.05*int(count_b))
            score=(score_vus+score_p+score_lp+score_b+score_lb)/(int(count_vus)+int(count_p)+int(count_lp)+int(count_b)+int(count_lb))
        
            clinvar_rawdata_vartype_reviewstatus_conflict_rename.iloc[rowindex,colindex]=score
            if (score>=75):
                clinvar_rawdata_vartype_reviewstatus_conflict_rename.iloc[rowindex,colindex_interpretation]="Conflicting interpretations of pathogenicity greater than or equal to 75"
            else:
                clinvar_rawdata_vartype_reviewstatus_conflict_rename.iloc[rowindex,colindex_interpretation]="Conflicting interpretations of pathogenicity less than 75"
     
        #concat PLP and conflicting:
        clinvar_rawdata_vartype_reviewstatus_PLPconflict_data = [clinvar_rawdata_vartype_reviewstatus_PLP_rename, clinvar_rawdata_vartype_reviewstatus_conflict_rename]
        clinvar_rawdata_vartype_reviewstatus_PLPconflict = pd.concat(clinvar_rawdata_vartype_reviewstatus_PLPconflict_data)
        TwoA_QC=len(clinvar_rawdata_vartype_reviewstatus_PLPconflict)

       
    
        clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter=clinvar_rawdata_vartype_reviewstatus_PLPconflict[clinvar_rawdata_vartype_reviewstatus_PLPconflict['SubmissionList'].str.contains("germline|de novo|maternal|paternal|uniparental|biparental|inherited")==True]
        TwoB_Originfilter=len(clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter)

        
        #filter 1 star review status
        Review_status_toexclude = ["no assertion criteria provided", "no assertion provided", "no assertion for the individual variant", "no classification provided","no classification for the individual variant"]
        clinvar_rawdata_vartype_reviewstatus_filter = clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter[~clinvar_rawdata_vartype_reviewstatus_PLPconflict_origin_filter['ReviewStatus'].isin(Review_status_toexclude)]
       
        Three_clinvarreviewstatus=len(clinvar_rawdata_vartype_reviewstatus_filter)
        
        clinvar_rawdata_vartype_reviewstatus=clinvar_rawdata_vartype_reviewstatus_filter
        # each variant is recorded once in ClinVar irrespective of number of times observed in individuals
        # estimate total germline events by counting germline submitters per variant ('Occurrence')
        # Occurrence calculation done by counting only 1 germline term per submitter (even if they had multiple submissions), if it exists- if variant does not have 'germline' term then 0 germline Occurrence and variant is filtered out (example: in the case of variants with only somatic or unknown submissions)
        # insert new column to calculate occurrence:
        clinvar_rawdata_vartype_reviewstatus.insert(1,"Occurrence","NaN")
        for index,row in clinvar_rawdata_vartype_reviewstatus.iterrows():
            rowindex=clinvar_rawdata_vartype_reviewstatus.index.get_loc(index)
            colindex=clinvar_rawdata_vartype_reviewstatus.columns.get_loc('Occurrence')
    
            originpacked=str(clinvar_rawdata_vartype_reviewstatus.loc[index]["SubmissionList"]).split("/")
            occurrence=0
            submissionlist=[]
            output_dict = {}
            for item in originpacked:
                submissiondetails=str(item).split("|")
                submissionlist.append(submissiondetails)
            submissionlist=submissionlist[:-1]
            for key, *values in submissionlist:
                if key not in output_dict:
                    output_dict[key] = [key]
                output_dict[key].extend(values)

            for item2 in output_dict.values():
                
                if ('germline' in item2) or ('de novo' in item2) or ('maternal' in item2) or ('paternal' in item2) or ('uniparental' in item2) or ('biparental' in item2) or ('inherited' in item2):
                    occurrence=occurrence+1
            clinvar_rawdata_vartype_reviewstatus.iloc[rowindex,colindex]=occurrence
            
           
        clinvar_rawdata_vartype_reviewstatus_occurrencegt0=clinvar_rawdata_vartype_reviewstatus[clinvar_rawdata_vartype_reviewstatus["Occurrence"]>0]
        
        Four_clinvarorigin=len(clinvar_rawdata_vartype_reviewstatus_occurrencegt0)
        
        #HGVSc filter: drop blanks (meaning no ref or alt info for the variant)
        clinvar_rawdata_vartype_reviewstatus_origin_filter=clinvar_rawdata_vartype_reviewstatus_occurrencegt0.dropna(subset=["HGVSc"])
        #HGVSc filter drop variants with '?' in HGVSc:
        clinvar_rawdata_vartype_reviewstatus_origin_filter.insert(1,"HGVScfilter","NaN")
        for index, row in clinvar_rawdata_vartype_reviewstatus_origin_filter.iterrows():
            rowindex=clinvar_rawdata_vartype_reviewstatus_origin_filter.index.get_loc(index)
            colindex=clinvar_rawdata_vartype_reviewstatus_origin_filter.columns.get_loc('HGVScfilter')
            if "?" not in clinvar_rawdata_vartype_reviewstatus_origin_filter.loc[index]['HGVSc']:
                clinvar_rawdata_vartype_reviewstatus_origin_filter.iloc[rowindex,colindex]="has no question mark"
        clinvar_rawdata_filter_HGVScfilter=clinvar_rawdata_vartype_reviewstatus_origin_filter[clinvar_rawdata_vartype_reviewstatus_origin_filter["HGVScfilter"].str.contains("has no question mark")==True]
        
        Five_HGVScfilter=len(clinvar_rawdata_filter_HGVScfilter)
        
        ##code block to account for vars with sq brackets in HGVSc '[xx]'
        clinvar_rawdata_filter_HGVScfilter.insert(1,"HGVSc_new","NaN")
        for index, row in clinvar_rawdata_filter_HGVScfilter.iterrows():
            rowindex=clinvar_rawdata_filter_HGVScfilter.index.get_loc(index)
            colindex=clinvar_rawdata_filter_HGVScfilter.columns.get_loc('HGVSc_new')
            if "[" not in clinvar_rawdata_filter_HGVScfilter.loc[index]['HGVSc']:
                clinvar_rawdata_filter_HGVScfilter.iloc[rowindex,colindex]= clinvar_rawdata_filter_HGVScfilter.loc[index]['HGVSc']
            else:
                if pd.isna(clinvar_rawdata_filter_HGVScfilter.loc[index]['HGVSg']):
                    clinvar_rawdata_filter_HGVScfilter.iloc[rowindex,colindex]= clinvar_rawdata_filter_HGVScfilter.loc[index]['HGVSc']
                else:
                    clinvar_rawdata_filter_HGVScfilter.iloc[rowindex,colindex]= clinvar_rawdata_filter_HGVScfilter.loc[index]['HGVSg']
               
        
        #HGVSc=clinvar_rawdata_filter_HGVScfilter["HGVSc"]
        #clinvar_processed_data_diffhgvsc=clinvar_rawdata_filter_HGVScfilter[~clinvar_rawdata_filter_HGVScfilter["HGVSc_new"].isin(HGVSc)]
 
        
        clinvar_rawdata_filter_HGVScfilter2=clinvar_rawdata_filter_HGVScfilter[~clinvar_rawdata_filter_HGVScfilter["HGVSc_new"].str.contains("\[")]
        clinvar_rawdata_filter_HGVScfilter2=clinvar_rawdata_filter_HGVScfilter2[~clinvar_rawdata_filter_HGVScfilter2["HGVSc_new"].str.contains("GRCh38")]
        
        FiveA=len(clinvar_rawdata_filter_HGVScfilter2)
        
        QC_lostvars=clinvar_rawdata_filter_HGVScfilter.drop(clinvar_rawdata_filter_HGVScfilter2.index)
        
        #print HGVSc for annotation of CA ID prior to cancer associated PLP variant filter
        clinvar_rawdata_filter_HGVScfilter2["HGVSc_new"].to_csv("Clinvar_filterPrePLP_HGVSctoannotate.txt", sep="\t", index=False, header=None)
        #print processed file for downstream analysis
        clinvar_rawdata_filter_HGVScfilter2.to_csv("Clinvar_filterPrePLP.txt", sep="\t")

        
        
        #filter PLP
        PLP = ["Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic"]
        clinvar_rawdata_vartype_reviewstatus_PLP= clinvar_rawdata_filter_HGVScfilter2[clinvar_rawdata_filter_HGVScfilter2['GL_first_Description'].str.contains("Pathogenic|Likely pathogenic|Pathogenic/Likely pathogenic")]
        Six_clinvarPLP=len(clinvar_rawdata_vartype_reviewstatus_PLP)
        
        Seven_clinvarconflictall=len(clinvar_rawdata_vartype_reviewstatus_conflict_rename)
        
        
        #filter for score greater than or equal to 75
        clinvar_rawdata_vartype_reviewstatus_conflict_gte75 = clinvar_rawdata_filter_HGVScfilter2[(clinvar_rawdata_filter_HGVScfilter2["Score"] >= 75)]
        Eight_clinvarconflictgte75=len(clinvar_rawdata_vartype_reviewstatus_conflict_gte75)
        
        #concat PLP and conflicting >=75:
        clinvar_rawdata_vartype_reviewstatus_PLPconflict_data = [clinvar_rawdata_vartype_reviewstatus_PLP, clinvar_rawdata_vartype_reviewstatus_conflict_gte75]
        clinvar_rawdata_vartype_reviewstatus_PLPconflict = pd.concat(clinvar_rawdata_vartype_reviewstatus_PLPconflict_data)
        print(len(clinvar_rawdata_vartype_reviewstatus_PLPconflict))
        Nine_clinvarPLPconflict=len(clinvar_rawdata_vartype_reviewstatus_PLPconflict)
        
        
        clinvar_rawdata_filter_PLPconflict_HGVScfilter=clinvar_rawdata_vartype_reviewstatus_PLPconflict
        Ten_occurrencetotal=clinvar_rawdata_filter_PLPconflict_HGVScfilter["Occurrence"].sum()
        Eleven_glsubmittertotal=clinvar_rawdata_filter_PLPconflict_HGVScfilter["GermlineClassificationNumberOfSubmitters"].astype(int).sum()
        Twelve_glsubmissiontotal=clinvar_rawdata_filter_PLPconflict_HGVScfilter["GermlineClassificationNumberOfSubmissions"].astype(int).sum()
        Thirteen_submittertotal=clinvar_rawdata_filter_PLPconflict_HGVScfilter["NumberOfSubmitters"].astype(int).sum()
        Fourteen_submissiontotal=clinvar_rawdata_filter_PLPconflict_HGVScfilter["NumberOfSubmissions"].astype(int).sum()
        
        #print HGVSc for VEP, CA ID annotation
        clinvar_rawdata_filter_PLPconflict_HGVScfilter["HGVSc_new"].to_csv("Clinvar_PLP_HGVSctoannotate.txt", sep="\t", index=False, header=None)
        
        #print processed file for downstream analysis (combine later with VEP and CA ID outputs)
        clinvar_rawdata_filter_PLPconflict_HGVScfilter.to_csv("Clinvar_PLP.txt", sep="\t")

        outputdf=pd.DataFrame({'Gene':[folder_name],'One_allclinvar': [One_allclinvar], 'Two_clinvarvartype':[Two_clinvarvartype], 'TwoA_QC':[TwoA_QC], 'TwoB_Originfilter':[TwoB_Originfilter],'Three_clinvarreviewstatus':[Three_clinvarreviewstatus], 'Four_clinvarorigin':[Four_clinvarorigin], 'Five_HGVScfilter':[Five_HGVScfilter],'FiveA_HGVScfiltersqbrackets':[FiveA], 'Six_clinvarPLP':[Six_clinvarPLP],'Seven_clinvarconflictall':[Seven_clinvarconflictall], 'Eight_clinvarconflictgte75':[Eight_clinvarconflictgte75], 'Nine_clinvarPLPconflict':[Nine_clinvarPLPconflict], 'Ten_occurrencetotal':[Ten_occurrencetotal], 'Eleven_glsubmittertotal':[Eleven_glsubmittertotal], 'Twelve_glsubmissiontotal':[Twelve_glsubmissiontotal], 'Thirteen_submittertotal':[Thirteen_submittertotal], 'Fourteen_submissiontotal':[Fourteen_submissiontotal]})
        allTSGdfcounts=pd.concat([allTSGdfcounts,outputdf])
        
        #checkpoint for where the code is while running- print gene name (directory)
        print("Current directory:", os.getcwd())
        #go back to parent directory after completing gene and start over with next gene
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)

# print file to a directory of your choice to see variant counts by gene at every filter step
#allTSGdfcounts.to_csv("Paper/AllTSG_df_counts_clinvar_extract_processed_codetopublishrun_7-24-25.txt", sep="\t")

<>:210: SyntaxWarning: invalid escape sequence '\['
<>:210: SyntaxWarning: invalid escape sequence '\['
/var/folders/84/djb4q50s33b6t8_cvx6h9src0000gn/T/ipykernel_51927/2770092401.py:210: SyntaxWarning: invalid escape sequence '\['
  clinvar_rawdata_filter_HGVScfilter2=clinvar_rawdata_filter_HGVScfilter[~clinvar_rawdata_filter_HGVScfilter["HGVSc_new"].str.contains("\[")]


Thu Jul 24 15:26:41 2025
WT1
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/WT1
94
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/WT1/Python_coderun_6-13-24_clinvar_extract_and_process_again
VHL
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/VHL
354
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/VHL/Python_coderun_6-13-24_clinvar_extract_and_process_again
TSC2
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/TSC2
1114
Current directory: /Users/suhasinilulla/Libra

In [3]:
allTSGdfcounts.describe()

,One_allclinvar,Two_clinvarvartype,TwoA_QC,TwoB_Originfilter,Three_clinvarreviewstatus,Four_clinvarorigin,Five_HGVScfilter,FiveA_HGVScfiltersqbrackets,Six_clinvarPLP,Seven_clinvarconflictall,Eight_clinvarconflictgte75,Nine_clinvarPLPconflict,Ten_occurrencetotal,Eleven_glsubmittertotal,Twelve_glsubmissiontotal,Thirteen_submittertotal,Fourteen_submissiontotal
count,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000
mean,5026.525000,4983.875000,4983.875000,4762.325000,4680.500000,4680.500000,4597.500000,4578.100000,828.950000,483.525000,13.600000,842.550000,1986.400000,2363.250000,2430.575000,2363.250000,2430.575000
std,4754.882565,4751.160491,4751.160491,4464.226343,4378.623651,4378.623651,4306.709173,4293.278693,1105.246485,894.183397,19.537997,1119.200423,3496.873731,3969.495239,4006.735533,3969.495239,4006.735533
min,337.000000,314.000000,314.000000,268.000000,256.000000,256.000000,253.000000,252.000000,28.000000,6.000000,0.000000,28.000000,32.000000,36.000000,36.000000,36.000000,36.000000
25%,2032.000000,1967.250000,1967.250000,1945.750000,1932.000000,1932.000000,1891.500000,1881.750000,175.000000,128.250000,1.000000,179.500000,299.750000,309.750000,312.000000,309.750000,312.000000
50%,3466.500000,3442.500000,3442.500000,3187.000000,3036.500000,3036.500000,2969.000000,2955.000000,393.500000,186.000000,6.000000,411.000000,806.000000,905.500000,908.000000,905.500000,908.000000
75%,5728.250000,5704.750000,5704.750000,5550.250000,5449.500000,5449.500000,5352.250000,5326.250000,894.500000,421.000000,13.000000,903.250000,2006.250000,2383.750000,2569.500000,2383.750000,2569.500000
max,18892.000000,18784.000000,18784.000000,18440.000000,18098.000000,18098.000000,17873.000000,17820.000000,4649.000000,5108.000000,78.000000,4693.000000,17104.000000,18963.000000,19035.000000,18963.000000,19035.000000


In [5]:
allTSGdfcounts.sum()

Gene                           WT1VHLTSC2TSC1TP53SUFUSTK11SMARCB1SMARCA4SMAD4...
One_allclinvar                                                            201061
Two_clinvarvartype                                                        199355
TwoA_QC                                                                   199355
TwoB_Originfilter                                                         190493
Three_clinvarreviewstatus                                                 187220
Four_clinvarorigin                                                        187220
Five_HGVScfilter                                                          183900
FiveA_HGVScfiltersqbrackets                                               183124
Six_clinvarPLP                                                             33158
Seven_clinvarconflictall                                                   19341
Eight_clinvarconflictgte75                                                   544
Nine_clinvarPLPconflict     